In [1]:
!pip install langgraph langchain langchain-openai python-dotenv requests typing
!pip install groq langchain-groq

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached tiktoken-0.12.0-cp312-cp312-win_amd64.whl.metadata (6.9 kB)
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 13.6 MB/s  0:00:00
Using cached tiktoken-0.12.0-cp312-cp312-win_amd64.whl (878 kB)
  Created wheel for typing: filename=typing-3.7.4.3-py3-none-any.whl size=26419 sha256=1723c843b345543ebb5ee50be4a379c3397dfeef1a764ebe30bcb5a881bc02b5
  Stored in directory: c:\users\rahul\appdata\local\pip\cache\wheels\12\98\52\2bffe242a9a487f00886e43b8ed8dac46456702e11a0d6abef
Successfully built typing

   ---------------------------------------- 0/4 [typing]
   ---------- ---------

In [2]:
!pip install pandas

In [3]:
import os 
import json
from typing import TypedDict, Optional, Dict, Any

In [4]:
import requests
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage

In [5]:
#Load environment variables

load_dotenv()

LTA_ACCOUNT_KEY = os.getenv("LTA_ACCOUNT_KEY")
if not LTA_ACCOUNT_KEY:
    raise ValueError("LTA_ACCOUNT_KEY is not in .env file")



In [6]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is not set in the .env file")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",  # Groq's LLaMA 3.3 70B model name
    temperature=0.2
)

In [7]:
class AgentState(TypedDict):
    #original user query
    user_query : str

    #parsed intent info
    intent: str     # ex - "bus_arrival"
    bus_stop_code: Optional[str] # ex - "83139" 
    bus_service: Optional[str]   # ex - "2"

    #control flags
    needs_api_call: bool

    #api related data
    api_raw_response: Optional[Dict[str, Any]]
    api_parsed_summary: Optional[str]

    #final agent answer
    final_answer: Optional[str]


# We’ll initialize this state each time we handle a query.

In [8]:
#talk to LTA Data Mall

BUS_ARRIVAL_URL = "https://datamall2.mytransport.sg/ltaodataservice/v3/BusArrival"

def get_bus_arrivals(
    bus_stop_code: str,
    bus_service: Optional[str] = None
) -> Dict[str, Any]:
    """
    Call the LTA DataMall BusArrivalv2 API for a given bus stop (and optionally a specific service).
    
    Returns a dict with:
    - "bus_stop_code"
    - "services": list of {service_no, next_arrivals: [eta_iso, est_wait_minutes]}
    - "raw": full JSON response from the API
    """
    headers = {
    "AccountKey": LTA_ACCOUNT_KEY,
        "accept": "application/json"
    }
    params = {
        "BusStopCode": bus_stop_code
    }
    response = requests.get(BUS_ARRIVAL_URL, headers=headers, params=params, timeout=10)

    if response.status_code != 200:
        raise RuntimeError(
            f"BusArrival API call failed with status {response.status_code}: {response.text[:200]}"
        )

    data = response.json()

    # Expect "Services" field in the response
    services = data.get("Services", [])

    # Optionally filter a specific service
    if bus_service:
        services = [s for s in services if s.get("ServiceNo") == bus_service]

    parsed_services = []

    from datetime import datetime, timezone

    for svc in services:
        service_no = svc.get("ServiceNo")
        # Each service has NextBus, NextBus2, NextBus3
        arrivals = []
        for key in ["NextBus", "NextBus2", "NextBus3"]:
            bus_info = svc.get(key, {})
            est_arrival = bus_info.get("EstimatedArrival")
            if not est_arrival:
                continue

            # LTA returns ISO timestamps, we can keep them as-is or compute minutes
            try:
                eta_dt = datetime.fromisoformat(est_arrival.replace("Z", "+00:00"))
                now_utc = datetime.now(timezone.utc)
                wait_minutes = int((eta_dt - now_utc).total_seconds() // 60)
            except Exception:
                wait_minutes = None

            arrivals.append({
                "eta_iso": est_arrival,
                "est_wait_minutes": wait_minutes
            })

        parsed_services.append({
            "service_no": service_no,
            "next_arrivals": arrivals
        })

    return {
        "bus_stop_code": bus_stop_code,
        "services": parsed_services,
        "raw": data
    }

In [9]:
# create empty/stub fn

def planner_node(state: AgentState) -> AgentState:
    """
    LLM-based planner.
    - Reads: state["user_query"]
    - Writes: intent, bus_stop_code, bus_service, needs_api_call
    """
    # We'll fill this in next step.
    raise NotImplementedError


def call_lta_node(state: AgentState) -> AgentState:
    """
    Tool node that calls the LTA BusArrival API using get_bus_arrivals.
    - Reads: bus_stop_code, bus_service
    - Writes: api_raw_response, api_parsed_summary
    """
    # We'll fill this in next step.
    raise NotImplementedError


def responder_node(state: AgentState) -> AgentState:
    """
    Response generator.
    - Reads: user_query, api_parsed_summary (or maybe intent only)
    - Writes: final_answer
    """
    # We'll fill this in next step.
    raise NotImplementedError


In [10]:
#Planner Node
# This node reads user_query and fills: intent, bus_stop_code, bus_service, needs_api_call


import json
from typing import cast

def planner_node(state: AgentState) -> AgentState:
    """
    LLM-based planner.
    - Reads: state["user_query"]
    - Writes: intent, bus_stop_code, bus_service, needs_api_call
    """

    user_query = state["user_query"]

    prompt = f"""
You are an intent classification and parameter extraction assistant for a Singapore public transport agent.

The user will ask questions about buses (e.g. arrival times, which buses are coming, etc.).

Your job:
1. Decide if the query is about bus arrivals at a specific bus stop.
2. If yes, extract:
   - bus_stop_code (a numeric string like "83139")
   - bus_service (if they mention a specific bus service number, like "2" or "10")
3. Decide whether we need to call the LTA BusArrival API.

Allowed intents:
- "bus_arrival"  -> the user wants real-time info about buses at a bus stop
- "chit_chat"    -> greetings, thanks, or general talk not requiring an API call

Always respond with STRICT JSON, no extra text, in the following format:

{{
  "intent": "<bus_arrival|chit_chat>",
  "bus_stop_code": "<string or null>",
  "bus_service": "<string or null>",
  "needs_api_call": <true or false>
}}

User query: "{user_query}"
"""

    response = llm.invoke(prompt)

    try:
        parsed = json.loads(response.content)
    except json.JSONDecodeError:
        # Fallback: be safe if parsing fails
        parsed = {
            "intent": "chit_chat",
            "bus_stop_code": None,
            "bus_service": None,
            "needs_api_call": False
        }

    # Update state
    state["intent"] = parsed.get("intent", "chit_chat")
    state["bus_stop_code"] = parsed.get("bus_stop_code")
    state["bus_service"] = parsed.get("bus_service")
    state["needs_api_call"] = bool(parsed.get("needs_api_call", False))

    return state


In [11]:
# LTA Tool Node (call API + summarize)
# This node uses get_bus_arrivals and prepares a simple text summary for the responder.

def call_lta_node(state: AgentState) -> AgentState:
    """
    Tool node that calls the LTA BusArrival API using get_bus_arrivals.
    - Reads: bus_stop_code, bus_service
    - Writes: api_raw_response, api_parsed_summary
    """

    bus_stop_code = state.get("bus_stop_code")
    bus_service = state.get("bus_service")

    if not bus_stop_code:
        # Can't call API if we don't know the bus stop.
        state["api_raw_response"] = None
        state["api_parsed_summary"] = "No bus stop code was provided, so the API could not be called."
        return state

    try:
        result = get_bus_arrivals(bus_stop_code, bus_service)
        state["api_raw_response"] = result

        services = result.get("services", [])
        if not services:
            summary = f"No upcoming bus arrivals found for stop {bus_stop_code}"
            if bus_service:
                summary += f" and service {bus_service}."
            else:
                summary += "."
        else:
            # Build a human-readable summary string
            lines = [f"Bus arrivals for stop {bus_stop_code}:"]
            for svc in services:
                s_no = svc.get("service_no")
                arrivals = svc.get("next_arrivals", [])
                if not arrivals:
                    lines.append(f"  - Service {s_no}: no estimated arrivals.")
                    continue

                eta_descriptions = []
                for arr in arrivals:
                    mins = arr.get("est_wait_minutes")
                    if mins is None:
                        eta_descriptions.append("ETA unknown")
                    elif mins <= 0:
                        eta_descriptions.append("arriving now")
                    else:
                        eta_descriptions.append(f"in {mins} minutes")

                joined_eta = ", ".join(eta_descriptions)
                lines.append(f"  - Service {s_no}: {joined_eta}")

            summary = "\n".join(lines)

        state["api_parsed_summary"] = summary

    except Exception as e:
        state["api_raw_response"] = None
        state["api_parsed_summary"] = f"Error while calling LTA BusArrival API: {e}"

    return state


In [12]:
# Responder Node (LLM: turn data into a nice answer)
# This node creates the final user-facing answer.

def responder_node(state: AgentState) -> AgentState:
    """
    Response generator.
    - Reads: user_query, intent, api_parsed_summary
    - Writes: final_answer
    """

    intent = state.get("intent", "chit_chat")
    user_query = state["user_query"]
    api_summary = state.get("api_parsed_summary")

    if intent == "chit_chat" or not state.get("needs_api_call", False):
        # Simple "chatty" answer without API
        prompt = f"""
You are a friendly Singapore public transport assistant.

The user asked: "{user_query}"

Their query does NOT require calling the live bus API (it's probably a greeting or generic question).
Reply briefly and helpfully in 1–3 sentences.
"""
        response = llm.invoke(prompt)
        state["final_answer"] = response.content
        return state

    # If we reach here, we used the API
    prompt = f"""
You are a Singapore public transport assistant.

The user asked:
"{user_query}"

Here is the structured information about bus arrivals (already parsed for you):

{api_summary}

Using ONLY this information:
- Answer the user's question clearly.
- Be concise (2–4 sentences).
- Mention bus stop code and services when relevant.
"""

    response = llm.invoke(prompt)
    state["final_answer"] = response.content

    return state


In [13]:
# Build the LangGraph Workflow
# Now we connect nodes into a graph:

from langgraph.graph import StateGraph, END

# Create the graph
graph = StateGraph(AgentState)

graph.add_node("planner", planner_node)
graph.add_node("call_lta", call_lta_node)
graph.add_node("responder", responder_node)

graph.set_entry_point("planner")


def route_after_planner(state: AgentState) -> str:
    """
    Decide whether to call the LTA API or go directly to responder.
    """
    if state.get("needs_api_call", False):
        return "call_lta"
    else:
        return "responder"


graph.add_conditional_edges(
    "planner",
    route_after_planner,
    {
        "call_lta": "call_lta",
        "responder": "responder"
    }
)

graph.add_edge("call_lta", "responder")
graph.add_edge("responder", END)

app = graph.compile()


In [14]:
# Helper to Run a Single Query
# This makes it easy to test and later reuse for multi-user simulation.

def run_agent_query(user_query: str, show_steps: bool = True) -> AgentState:
    """
    Run the LangGraph agent for a single user query.
    Returns the final state.

    Uses stream_mode="values" so each event is the full AgentState.
    """

    # Initial state
    state: AgentState = {
        "user_query": user_query,
        "intent": "",
        "bus_stop_code": None,
        "bus_service": None,
        "needs_api_call": False,
        "api_raw_response": None,
        "api_parsed_summary": None,
        "final_answer": None,
    }

    final_state: AgentState | None = None

    # NOTE: stream_mode="values" is the key fix here 👇
    for event in app.stream(state, stream_mode="values"):
        final_state = event

        if show_steps:
            print("---- Step ----")
            print({
                "intent": final_state.get("intent"),
                "bus_stop_code": final_state.get("bus_stop_code"),
                "bus_service": final_state.get("bus_service"),
                "needs_api_call": final_state.get("needs_api_call"),
                "api_parsed_summary": final_state.get("api_parsed_summary"),
                "final_answer": final_state.get("final_answer"),
            })
            print()

    if final_state is None:
        raise RuntimeError("Agent did not produce any state (graph may not have run).")

    return final_state



In [15]:
def ask_agent(user_query: str) -> str:
    final_state = run_agent_query(user_query, show_steps=False)
    return final_state["final_answer"]


In [16]:
final = run_agent_query("When is the next bus arriving at stop 20251?")
print("\nFinal answer:\n", final["final_answer"])


---- Step ----
{'intent': '', 'bus_stop_code': None, 'bus_service': None, 'needs_api_call': False, 'api_parsed_summary': None, 'final_answer': None}

---- Step ----
{'intent': 'bus_arrival', 'bus_stop_code': '20251', 'bus_service': None, 'needs_api_call': True, 'api_parsed_summary': None, 'final_answer': None}

---- Step ----
{'intent': 'bus_arrival', 'bus_stop_code': '20251', 'bus_service': None, 'needs_api_call': True, 'api_parsed_summary': 'No upcoming bus arrivals found for stop 20251.', 'final_answer': None}

---- Step ----
{'intent': 'bus_arrival', 'bus_stop_code': '20251', 'bus_service': None, 'needs_api_call': True, 'api_parsed_summary': 'No upcoming bus arrivals found for stop 20251.', 'final_answer': 'Unfortunately, there are no upcoming bus arrivals found for stop 20251. You may want to check the bus schedule or plan your route again. If you need assistance with alternative travel options, feel free to ask.'}


Final answer:
 Unfortunately, there are no upcoming bus arrivals

In [17]:
final = run_agent_query("When is the next bus arriving at stop 84009?")
print("\nFinal answer:\n", final["final_answer"])


---- Step ----
{'intent': '', 'bus_stop_code': None, 'bus_service': None, 'needs_api_call': False, 'api_parsed_summary': None, 'final_answer': None}

---- Step ----
{'intent': 'bus_arrival', 'bus_stop_code': '84009', 'bus_service': None, 'needs_api_call': True, 'api_parsed_summary': None, 'final_answer': None}

---- Step ----
{'intent': 'bus_arrival', 'bus_stop_code': '84009', 'bus_service': None, 'needs_api_call': True, 'api_parsed_summary': 'No upcoming bus arrivals found for stop 84009.', 'final_answer': None}

---- Step ----
{'intent': 'bus_arrival', 'bus_stop_code': '84009', 'bus_service': None, 'needs_api_call': True, 'api_parsed_summary': 'No upcoming bus arrivals found for stop 84009.', 'final_answer': 'Unfortunately, there are no upcoming bus arrivals found for stop 84009. You may want to check the bus schedule or plan your route in advance. Currently, there is no estimated arrival time available for buses at this stop.'}


Final answer:
 Unfortunately, there are no upcoming b

In [18]:
queries = [
    "When is the next bus arriving at stop 20251?",
    "What buses are coming to stop 20251 in the next 10 minutes?",
    "Is bus 2 arriving at stop 20251 soon?",
    "What are the arrival times for all buses at stop 75009?",
    "Is there any bus arriving at stop 75009 within 5 minutes?",
    "When will bus 10 arrive at stop 83301?",
    "What is the next service at bus stop 45009?",
    "Give me all upcoming buses at 97029.",
    "How long do I have to wait for bus 5 at stop 20251?",
    "Are there any buses approaching stop 97029 right now?"
]

multi_results = []

for i, q in enumerate(queries, start=1):
    print(f"=== User {i} ===")
    print("Query:", q)
    final_state = run_agent_query(q, show_steps=False)
    answer = final_state["final_answer"]
    print("Answer:", answer)
    print()

    multi_results.append({
        "user_id": f"user_{i}",
        "query": q,
        "response": answer
    })


=== User 1 ===
Query: When is the next bus arriving at stop 20251?
Answer: Unfortunately, there are no upcoming bus arrivals found for stop 20251. You may want to check the bus schedule or plan your route again. If you need assistance with alternative bus services or routes, please let me know and I'll be happy to help.

=== User 2 ===
Query: What buses are coming to stop 20251 in the next 10 minutes?
Answer: There are no upcoming bus arrivals at stop 20251 in the next 10 minutes. Unfortunately, we couldn't find any buses scheduled to arrive at this stop within the specified time frame. If you'd like to plan your journey, I can try to assist you with alternative options or provide more information on stop 20251.

=== User 3 ===
Query: Is bus 2 arriving at stop 20251 soon?
Answer: Unfortunately, there are no upcoming bus arrivals for service 2 at stop 20251. You won't be seeing bus 2 arriving at this stop soon. If you'd like to check for other bus services, I can try to assist you with 

In [19]:
import pandas as pd

df_results = pd.DataFrame(multi_results)
df_results


,user_id,query,response
0,user_1,When is the next bus arriving at stop 20251?,"Unfortunately, there are no upcoming bus arriv..."
1,user_2,What buses are coming to stop 20251 in the nex...,There are no upcoming bus arrivals at stop 202...
2,user_3,Is bus 2 arriving at stop 20251 soon?,"Unfortunately, there are no upcoming bus arriv..."
3,user_4,What are the arrival times for all buses at st...,There are no upcoming bus arrivals found for b...
4,user_5,Is there any bus arriving at stop 75009 within...,"Unfortunately, there are no upcoming bus arriv..."
5,user_6,When will bus 10 arrive at stop 83301?,"Unfortunately, there are no upcoming bus arriv..."
6,user_7,What is the next service at bus stop 45009?,There are no upcoming bus arrivals found for b...
7,user_8,Give me all upcoming buses at 97029.,"At bus stop 97029, there are no upcoming bus a..."
8,user_9,How long do I have to wait for bus 5 at stop 2...,"Unfortunately, there are no upcoming bus arriv..."
9,user_10,Are there any buses approaching stop 97029 rig...,There are no buses approaching bus stop 97029 ...
